In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix

In [2]:
df=pd.read_csv(r'C:\Users\user\OneDrive\Desktop\5thsem_ojt\project1_diabetes\data\processed\hebin_p1w2_features.csv')

In [3]:
feature_cols = ["age","pregnancies","glucose","blood_pressure",
                "skin_thickness","insulin","bmi","diabetes_pedigree",
                "bmi_category","glucose_band"]
x = df[feature_cols].copy()
y = df["outcome"]
x

,age,pregnancies,glucose,blood_pressure,skin_thickness,insulin,bmi,diabetes_pedigree,bmi_category,glucose_band
0,79,0,124.000000,71.0,30.0,124.0,23.4,0.588,normal,prediabetic
1,37,0,153.000000,85.0,24.0,42.0,33.9,0.192,obese,diabetic
2,39,0,142.000000,68.0,22.0,159.0,26.9,0.777,overweight,diabetic
3,68,7,121.000000,88.0,26.0,124.0,29.0,1.217,overweight,prediabetic
4,75,0,107.000000,84.0,31.0,101.0,21.2,0.278,normal,prediabetic
...,...,...,...,...,...,...,...,...,...,...
945,64,2,117.855165,81.0,34.0,124.0,40.3,0.641,obese,prediabetic
946,71,0,140.000000,64.0,37.0,124.0,33.4,0.116,obese,diabetic
947,40,0,130.000000,73.0,26.0,258.0,20.1,0.709,normal,diabetic
948,52,4,123.000000,76.0,22.0,189.0,38.9,0.540,obese,prediabetic


In [4]:
x=pd.get_dummies(x,drop_first=True)
x.shape

(950, 13)

In [5]:
x_train,x_test,y_train,y_test= train_test_split(
    x,y,test_size=0.2,random_state=42,stratify=y)

In [6]:
scaler = StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)#Calculates the mean and standard deviation from x train
x_test_scaled=scaler.transform(x_test)#Uses those values to standardize x_train.

In [7]:
logreg = LogisticRegression(max_iter=1000).fit(x_train_scaled, y_train)
dtree = DecisionTreeClassifier(max_depth=3,random_state=42).fit(x_train, y_train)
knn = KNeighborsClassifier(n_neighbors=25).fit(x_train_scaled, y_train)



In [8]:
print('LogisticRegression',accuracy_score(y_test,logreg.predict(x_test_scaled)))
print('DecisionTreeClassifier',accuracy_score(y_test,dtree.predict(x_test)))
print('kneighborsclassifier',accuracy_score(y_test,knn.predict(x_test_scaled)))

LogisticRegression 0.7526315789473684
DecisionTreeClassifier 0.7210526315789474
kneighborsclassifier 0.7473684210526316


In [9]:
models = [
    ("Logistic Regression", logreg, x_test_scaled),
    ("Decision Tree (d=3)", dtree, x_test),
    ("KNN (k=25)", knn, x_test_scaled)
]

for name, m, Xt in models:
    print(f"{name:22s} accuracy = {accuracy_score(y_test, m.predict(Xt)):.4f}")

print("\nTest set:", len(y_test), "patients |",
      int(np.sum(y_test)), "of them diabetic")

Logistic Regression    accuracy = 0.7526
Decision Tree (d=3)    accuracy = 0.7211
KNN (k=25)             accuracy = 0.7474

Test set: 190 patients | 55 of them diabetic


In [10]:
def cm_row(name,model,xt):
   cm=confusion_matrix(y_test,model.predict(xt))
   tn,fp,fn,tp=cm.ravel()
   return{"model":name,"correct_negatives":int(tn),"false_alarams":int(fp),
          "missed_patients":int(fn),"found_patients":int(tp),
          "accuracy": round(model.score(xt, y_test), 4)
    } 
pd.DataFrame([
    cm_row("Logistic Regression",logreg, x_test_scaled),
    cm_row("Decision Tree (d=3)", dtree, x_test),
    cm_row("KNN (k=25)", knn, x_test_scaled),
])

,model,correct_negatives,false_alarams,missed_patients,found_patients,accuracy
0,Logistic Regression,127,8,39,16,0.7526
1,Decision Tree (d=3),121,14,39,16,0.7211
2,KNN (k=25),132,3,45,10,0.7474


Build a confusion matrices for every model

In [11]:
rows = []
for name,m,Xt in models:
    cm = confusion_matrix(y_test,m.predict(Xt))
    tn, fp, fn, tp = cm.ravel()
    rows.append({"models": name, "correct_negative": int(tn),"False_alarms":int(fp),
                 "missed_patients":int(fn),"found_patients":int(tp),
                 "accuracy":accuracy_score(y_test,m.predict(Xt))})
    metrices = pd.DataFrame(rows)
metrices

,models,correct_negative,False_alarms,missed_patients,found_patients,accuracy
0,Logistic Regression,127,8,39,16,0.752632
1,Decision Tree (d=3),121,14,39,16,0.721053
2,KNN (k=25),132,3,45,10,0.747368


Correct Negative (TN) = 127 → 127 patients were correctly predicted as negative.
False Alarm (FP) = 8 → 8 patients were predicted positive, but were actually negative.
Missed Patients (FN) = 39 → 39 patients actually had the disease, but the model predicted negative.
Found Patients (TP) = 16 → 16 patients with the disease were correctly identified.
Accuracy = 75.26%

Logistic Regression is dominated by missed patients, Decision Tree is dominated by false alarms, and KNN is dominated by missed patients

Score the model that never says yes

In [12]:
from sklearn.dummy import DummyClassifier

dummy=DummyClassifier(strategy='most_frequent')
dummy.fit(x_train,y_train)
y_pred=dummy.predict(x_test)
confusion_matrix(y_test,y_pred)

array([[135,   0],
       [ 55,   0]])

In [13]:
print("Dummy Classifier Accuracy:", accuracy_score(y_test, y_pred))

Dummy Classifier Accuracy: 0.7105263157894737


In [14]:
y_test.value_counts(normalize=True)

outcome
0    0.710526
1    0.289474
Name: proportion, dtype: float64

# PRECISION AND RECALL

Recall:
Recall tells us how many of the actual positive cases the model successfully found.
Ability to find positive cases
EQUATION=TP/TP+FN

In [15]:
cm =confusion_matrix(y_test,logreg.predict(x_test_scaled))
tn,fp,fn,tp=cm.ravel()
print("recall:tp/tp+fn=",(tp/(tp+fn)))


recall:tp/tp+fn= 0.2909090909090909


In [16]:
from sklearn.metrics import recall_score 



In [17]:
recall_score(y_test,logreg.predict(x_test_scaled))
print("LogisticRegression",recall_score(y_test,logreg.predict(x_test_scaled)))
print("DecisionTreeClassifier  Recall =",recall_score(y_test,dtree.predict(x_test)))
print("KNeighborsClassifier  Recall =",recall_score(y_test,knn.predict(x_test_scaled)))

LogisticRegression 0.2909090909090909
DecisionTreeClassifier  Recall = 0.2909090909090909
KNeighborsClassifier  Recall = 0.18181818181818182


# PRECISION

In [18]:
cm =confusion_matrix(y_test,logreg.predict(x_test_scaled))
tn,fp,fn,tp=cm.ravel()
print("precision:tp/tp+fp=",(tp/(tp+fp)))

precision:tp/tp+fp= 0.6666666666666666


In [19]:
#Precision:
#Precision tells us how many of the positive predictions made by the model were actually correct.
#EQUATION:tp/tp+fp

In [20]:
from sklearn.metrics import precision_score

In [21]:
precision_score(y_test,logreg.predict(x_test_scaled))
print("LogisticRegression",precision_score(y_test,logreg.predict(x_test_scaled)))
print("DecisionTreeClassifier  precision =",precision_score(y_test,dtree.predict(x_test)))
print("KNeighborsClassifier  precision =",precision_score(y_test,knn.predict(x_test_scaled)))

LogisticRegression 0.6666666666666666
DecisionTreeClassifier  precision = 0.5333333333333333
KNeighborsClassifier  precision = 0.7692307692307693


Build the three metric table 

In [25]:
rows=[]
for name,m,xt in models:
    pred = m.predict(xt)
    cm=confusion_matrix(y_test,pred)
    tn,fp,fn,tp=cm.ravel()
    rows.append({
              "model":name,
              "recall":round(recall_score(y_test,pred,zero_division=0),4),
              "precision":round(precision_score(y_test,pred,zero_division=0),4),
              "accuracy":round(accuracy_score(y_test,pred),4),
              "correct_negatives":int(tn),"false_alarams":int(fp),
              "missed_patients":int(fn),"found_patients":int(tp)})
    
results=pd.DataFrame(rows)

In [26]:
results

,model,recall,precision,accuracy,correct_negatives,false_alarams,missed_patients,found_patients
0,Logistic Regression,0.2909,0.6667,0.7526,127,8,39,16
1,Decision Tree (d=3),0.2909,0.5333,0.7211,121,14,39,16
2,KNN (k=25),0.1818,0.7692,0.7474,132,3,45,10


Sort by precision

In [27]:
results.sort_values('precision', ascending=False)

,model,recall,precision,accuracy,correct_negatives,false_alarams,missed_patients,found_patients
2,KNN (k=25),0.1818,0.7692,0.7474,132,3,45,10
0,Logistic Regression,0.2909,0.6667,0.7526,127,8,39,16
1,Decision Tree (d=3),0.2909,0.5333,0.7211,121,14,39,16


Sort by recall

In [28]:
results.sort_values('recall',ascending=False)

,model,recall,precision,accuracy,correct_negatives,false_alarams,missed_patients,found_patients
0,Logistic Regression,0.2909,0.6667,0.7526,127,8,39,16
1,Decision Tree (d=3),0.2909,0.5333,0.7211,121,14,39,16
2,KNN (k=25),0.1818,0.7692,0.7474,132,3,45,10


In [30]:
probs=logreg.predict_proba(x_test_scaled)[:,1]
preds=logreg.predict(x_test_scaled)

pd.DataFrame({'probability': probs[:10].round(3),
              'prediction': preds[:10],
              'actual':  np.asarray(y_test)[:10]})


,probability,prediction,actual
0,0.289,0,0
1,0.032,0,0
2,0.182,0,0
3,0.182,0,0
4,0.177,0,1
5,0.161,0,0
6,0.419,0,0
7,0.473,0,1
8,0.286,0,0
9,0.121,0,0


In [32]:
sweep=[]
for t in [0.5,0.45,0.4,0.35,0.3,0.25,0.2,0.15]:
    pred_t=(probs>=t).astype(int)
    cm=confusion_matrix(y_test,pred_t)
    tn,fp,fn,tp=cm.ravel()

    sweep.append({
            "threshold":t,
            "accuracy": round(accuracy_score(y_test, pred_t), 4),
            "precision": round(precision_score(y_test, pred_t), 4),
            "recall": round(recall_score(y_test, pred_t), 4),
            "found": int(tp),
            "missed": int(fn),
            "false_alarms": int(fp),
            "flagged": int(tp+fp)
        })
    
pd.DataFrame(sweep)

,threshold,accuracy,precision,recall,found,missed,false_alarms,flagged
0,0.50,0.7526,0.6667,0.2909,16,39,8,24
1,0.45,0.7316,0.5556,0.3636,20,35,16,36
2,0.40,0.7105,0.5000,0.4364,24,31,24,48
3,0.35,0.7158,0.5082,0.5636,31,24,30,61
4,0.30,0.6947,0.4789,0.6182,34,21,37,71
5,0.25,0.6158,0.4043,0.6909,38,17,56,94
6,0.20,0.5789,0.3874,0.7818,43,12,68,111
7,0.15,0.5316,0.3692,0.8727,48,7,82,130


# F1 score

F1 Score is a metric that combines Precision and Recall into one value. It tells us how well the model balances correct positive predictions and finding actual positive cases
EQUATION:F1 = 2 × (Precision × Recall) / (Precision + Recall)

In [33]:
from sklearn.metrics import f1_score

In [36]:
print(f"f1_score_logistic regression={f1_score(y_test,logreg.predict(x_test_scaled)):.4f}"),
print(f"f1_score_decisiontree={f1_score(y_test,dtree.predict(x_test)):.4f}"),
print(f"f1_score_KNN={f1_score(y_test,knn.predict(x_test_scaled)):.4f}")

f1_score_logistic regression=0.4051
f1_score_decisiontree=0.3765
f1_score_KNN=0.2941


In [37]:
from sklearn.metrics import f1_score, classification_report
f1_score(y_test,pred)
print(classification_report(y_test,pred,digits=4))

              precision    recall  f1-score   support

           0     0.7458    0.9778    0.8462       135
           1     0.7692    0.1818    0.2941        55

    accuracy                         0.7474       190
   macro avg     0.7575    0.5798    0.5701       190
weighted avg     0.7526    0.7474    0.6864       190

